In [1]:
from utils.read_jsonl import read_jsonl
from datasets import Dataset
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, DistilBertTokenizer, DistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
import torch

In [2]:
# Detect MPS (Apple Silicon GPU)
force_cpu = False
device = torch.device("mps" if torch.backends.mps.is_available() and not force_cpu else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
df = read_jsonl("../DB-bio/combined_train_and_train_sft_anonymized.jsonl")

In [4]:
df.shape, df.columns

((3876, 2), Index(['text', 'label'], dtype='object'))

In [5]:
test_size = 0.7

# train/test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(), df["label"].tolist(), test_size=test_size, random_state=42
)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

In [6]:
use_distillbert = True

In [7]:
# Tokenization
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased") if use_distillbert else BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")

Map:   0%|          | 0/1162 [00:00<?, ? examples/s]

Map:   0%|          | 0/2714 [00:00<?, ? examples/s]

In [8]:
# Load model and move to MPS
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2) if use_distillbert else BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [9]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = torch.argmax(torch.tensor(logits), axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    correct = (preds == torch.tensor(labels)).sum().item()
    total = len(labels)
    print(f"\nCorrect predictions: {correct} / {total}")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

# Training arguments
training_args = TrainingArguments(
    use_cpu= force_cpu,
    dataloader_pin_memory=False,  # suppress pin_memory warning
    disable_tqdm=True,
    output_dir="./results/distilbert" if use_distillbert else "./results/bert",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=5e-6,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=4,
    eval_strategy="steps",
    eval_steps=16,
    report_to="none",
    fp16=False  # disable fp16 for MPS
)

In [10]:
# Custom Trainer to support MPS
class MPSTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        print(device)
        model.to(device)

    def _prepare_inputs(self, inputs):
        # Force inputs to MPS
        return {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}

# Use custom Trainer
trainer = MPSTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

cpu


In [11]:
trainer.train() 

{'loss': 0.6869, 'grad_norm': 1.0619584321975708, 'learning_rate': 4.594594594594596e-06, 'epoch': 0.10810810810810811}
{'loss': 0.6691, 'grad_norm': 1.262570858001709, 'learning_rate': 4.0540540540540545e-06, 'epoch': 0.21621621621621623}
{'loss': 0.6516, 'grad_norm': 1.4352939128875732, 'learning_rate': 3.513513513513514e-06, 'epoch': 0.32432432432432434}
{'loss': 0.6135, 'grad_norm': 1.7832036018371582, 'learning_rate': 2.9729729729729736e-06, 'epoch': 0.43243243243243246}

Correct predictions: 2605 / 2714
{'eval_loss': 0.6069008708000183, 'eval_accuracy': 0.9598378776713338, 'eval_precision': 0.9282263630089717, 'eval_recall': 0.9962962962962963, 'eval_f1': 0.961057520543051, 'eval_runtime': 137.5358, 'eval_samples_per_second': 19.733, 'eval_steps_per_second': 0.16, 'epoch': 0.43243243243243246}
{'loss': 0.6121, 'grad_norm': 1.9221045970916748, 'learning_rate': 2.432432432432433e-06, 'epoch': 0.5405405405405406}
{'loss': 0.5876, 'grad_norm': 2.0413615703582764, 'learning_rate': 1.8

TrainOutput(global_step=37, training_loss=0.6074570997341259, metrics={'train_runtime': 499.3255, 'train_samples_per_second': 2.327, 'train_steps_per_second': 0.074, 'train_loss': 0.6074570997341259, 'epoch': 1.0})